# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² clinical colorectal cancer dataset using the `mlcroissant` library. The workflow covers metadata loading, schema exploration, record set extraction, basic preprocessing, and visualization.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json), which provides standardized metadata and access to the data package.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print high-level metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Review the available record sets, fields, and their `@id`s as defined by the schema.

In [ ]:
# List all available record sets and their fields using their @id
print("Available record sets and their fields (by @id):\n")
record_sets = []
field_dict = {} # {record_set_id: [field_ids]}
for rs in dataset.record_sets:
    print(f"Record set: {rs['@id']} -- {rs.get('name', '')}")
    record_sets.append(rs['@id'])
    fields = rs.get('field', [])
    # Ensure always a list
    if isinstance(fields, dict):
        fields = [fields]
    field_ids = []
    for field in fields:
        # If it is a reference, use the reference. If an object, get its @id.
        if isinstance(field, dict):
            field_id = field.get('@id', str(field))
        else:
            field_id = str(field)
        print(f"  - field: {field_id}")
        field_ids.append(field_id)
    field_dict[rs['@id']] = field_ids
if record_sets:
    print("\nExample preview of records from the first record set:")
    example_id = record_sets[0]
    count = 0
    for rec in dataset.records(record_set=example_id):
        print(rec)
        count += 1
        if count >= 2:
            break

## 3. Data Extraction
Load data from each record set into a `pandas.DataFrame` for analysis. All references use the `@id` fields from the schema. For practical purposes, we extract all top-level record sets into individual DataFrames indexed by their `@id`.

In [ ]:
# Extract all record sets into DataFrames using their @id
dataframes = {}

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set {record_set_id}")
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")
        continue

if record_sets:
    # Show columns and head for the main record set (by convention, first one)
    main_rs_id = record_sets[0]
    print(f"\nColumns in record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, grouping, and preparing for further analysis. Here, we'll illustrate these on a numeric field from the main record set.

In [ ]:
# For demonstration, identify a numeric field in the main record set (e.g., 'Age_at_Second_CRC')
# You may change this field depending on schema inspection.

# Try to pick a common numeric field by scanning columns
main_rs_id = record_sets[0]
df = dataframes[main_rs_id]
possible_numeric_fields = [c for c in df.columns if c.lower().startswith(('age', 'interval', 'years')) or pd.api.types.is_numeric_dtype(df[c])]

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Using '{numeric_field}' as a numeric field for demo.")
else:
    # Fallback to the first column
    numeric_field = df.columns[0]
    print(f"No obvious numeric column, using '{numeric_field}'.")

# Attempt numeric conversion (in case the field is object type)
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

# Define a threshold for filtering (e.g., Age > 50)
threshold = 50
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where {numeric_field} > {threshold} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalization
if len(filtered_df) > 0:
    mean = filtered_df[numeric_field].mean()
    std = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
    print(f"\nFirst few normalized {numeric_field} values:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping by a likely categorical field, e.g., 'Sex' or 'Anatomical_Location' if available
likely_group_fields = [c for c in df.columns if ('sex' in c.lower() or 'location' in c.lower() or 'site' in c.lower()) and df[c].nunique() > 1]
if likely_group_fields:
    group_field = likely_group_fields[0]
    print(f"\nGrouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(grouped_df)
else:
    print("\nNo suitable group field found for grouping demonstration.")

## 5. Visualization
Visualize the distribution of the numeric field and compare grouped means if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot/group plot if grouping field is available
if 'group_field' in locals() and group_field and group_field in df.columns:
    plt.figure(figsize=(7, 4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
This notebook loaded the FAIR² colorectal cancer survivorship dataset via its Croissant schema, reviewed metadata and schema components using unique `@id` fields, and demonstrated data extraction, normalization, and basic visualization techniques with `mlcroissant`. This workflow provides a reproducible foundation for further clinical analytics and FAIR data re-use.